# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is an attribute object, not a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, let us enumerate all available Record Sets and their IDs.

In [ ]:
# Display all record sets, their @id, and included field @ids
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields in this RecordSet:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()

Let's preview the first record of the main record set, using its `@id`.

In [ ]:
# Display one record from each record set for inspection
for rs in record_sets:
    print(f"First record from RecordSet '{rs.name}' (@id: {rs.id}):")
    recs = dataset.records(record_set=rs.id)
    try:
        rec = next(recs)
        print(json.dumps(rec, indent=2))
    except StopIteration:
        print("  [No records found]")
    print("-"*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract all record sets by their `@id`.

In [ ]:
# Extract data from each record set
all_record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Choose a main record set for demonstration
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if main_record_set_id:
    print(f"Columns in main record set (@id: {main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations such as removing outliers, transforming distributions, and grouping by key attributes.

First, let's identify numeric fields in the main data record set.

In [ ]:
import numpy as np

if main_record_set_id and not dataframes[main_record_set_id].empty:
    # List numeric columns (by dtype)
    df = dataframes[main_record_set_id]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields in record set (@id: {main_record_set_id}): {numeric_cols}")

    # Default to the first numeric field for demo
    if numeric_cols:
        numeric_field = numeric_cols[0]
    else:
        numeric_field = df.columns[0]  # fallback if none detected
    print(f"Using numeric field: {numeric_field}")

    # Filtering: e.g., keep rows where value is above 10
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df = df[df[numeric_field] > threshold].copy()
    else:
        # Try to convert to numeric (in case data type is object)
        filtered_df = df.copy()
        filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalization
    if not filtered_df.empty:
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

    # Group by another field (try a likely category field)
    # Heuristically search for a likely candidate: string column with < 20 unique categories
    candidate_group_fields = [
        col for col in df.columns
        if df[col].dtype == object and df[col].nunique() < 20
    ]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Grouping data by: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df)
    else:
        print("No suitable group field found (small number of categories).")
else:
    print("No main record set with data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field for the filtered records.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' (> {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field found, plot normalized value by group
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[norm_col])
        plt.title(f"Normalized '{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(norm_col)
        plt.show()
else:
    print("No data or no numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This dataset contains mass spectrometry records of proteins isolated from human mast cells' extracellular vesicles.
- We loaded and previewed all record sets and fields by their Croissant `@id` identifiers.
- EDA was performed by filtering, normalizing, and grouping numeric protein data fields, and visualizations showed distributions and categorical differences (where possible).
- The dataset structure allows for analysis of protein abundance, modifications, and characteristics by leveraging the Croissant schema.